# Runtime & Space Complexity - Feature Engineering

Goes through the O-notation runtime of each feature group in `DenseFeatureTransformer`, then combines them at the end.

-------

## Notation

| Symbol | Meaning |
|--------|---------|
| $n$ | number of samples (rows) |
| $p = 32$ | total number of features (fixed by `DenseFeatureTransformer`) |
| $L$ | comment length in characters; since words $W \leq L$ and sentences $S \leq L$, all per-row costs reduce to $O(L)$ |

All feature functions operate on a one comment string. The outer loop over all $n$ rows is covered at the end.


## 0. Data Split - `split_and_features/prepare_split.py`

```python
# prepare_split.py, lines 40-44
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y,
)
```

- **`stratify=y`:** one pass over $n$ labels to count class frequencies -> $O(n)$
- **Data copy:** one pass over $n$ rows to fill train/test arrays -> $O(n)$
- No feature computation here -> $X$ is still raw comment text

**Time: $O(n)$; Space: $O(n)$**

---

## 1. Sentiment Features (6 features)
`vader_compound`, `vader_neg`, `vader_pos`, `vader_is_negative`, `vader_intensity`, `vader_pos_minus_neg`

```python
# build_features.py, lines 260-271
def _extract_sentiment(text: str, _sia=SentimentIntensityAnalyzer()) -> dict:
    scores   = _sia.polarity_scores(text)   # scans every token against VADER lexicon
    compound = scores["compound"]
    neg, pos = scores["neg"], scores["pos"]
    return {
        "vader_compound":      compound,
        "vader_neg":           neg,
        "vader_pos":           pos,
        "vader_is_negative":   int(compound < -0.05),   # O(1) threshold check
        "vader_intensity":     abs(compound),            # O(1)
        "vader_pos_minus_neg": round(pos - neg, 6),     # O(1)
    }
```

- **`_sia.polarity_scores(text)`:** tokenises text, linear scan over $L$ characters; looks up each token in the VADER lexicon (as a hash map) -> $O(1)$ per token; applies valence modifier rules (negation, punctuation, capitalisation) -> overall $O(L)$
- All 6 return values are $O(1)$ arithmetic on the already-computed VADER scores
- `SentimentIntensityAnalyzer` instantiated once when module is  loaded 

**Per row: $O(L)$; dominant operation => VADER token loop**


## 2. Second-Person Pronoun Features (3 features)
`has_second_person`, `second_person_count`, `second_person_density`

```python
# build_features.py, lines 275-283
_SECOND_PERSON_RE = re.compile(
    r"\b(you|your|yours|yourself|yourselves|you're|you'll|you've|you'd|ur|u)\b",
    re.IGNORECASE,
)  # compiled once at module level - line 92

def _extract_second_person(text: str) -> dict:
    matches    = _SECOND_PERSON_RE.findall(text)   # one regex scan: O(L)
    count      = len(matches)                       # O(1)
    word_count = max(len(text.split()), 1)          # O(L)
    return {
        "has_second_person":     int(count > 0),           # O(1)
        "second_person_count":   count,                    # O(1)
        "second_person_density": round(count / word_count, 6),  # O(1)
    }
```

- **`_SECOND_PERSON_RE.findall(text)`:** single linear scan (over  comment) -> $O(L)$
- **`text.split()`:** one pass over $L$ characters to split on whitespace -> $O(L)$
- All three return values are $O(1)$ 
- Regex compiled once at module level (line 92) 

**Per row: $O(L)$ (regex scan dominates**


## 3. Profanity Feature (1 feature)
`profanity_count`

```python
# build_features.py, lines 291-293
_WORD_RE = re.compile(r"\b[a-zA-Z']+\b")  # compiled once at module level - line 110

def _profanity_count(text: str) -> int:
    tokens = _WORD_RE.findall(text.lower())              # tokenise: O(L)
    return sum(1 for t in tokens if t in _PROFANITY_LEXICON)  # frozenset lookup: O(1) each
```

- **`_WORD_RE.findall(text.lower())`:** one regex scan over $L$ characters -> $O(L)$; produces $W$ tokens
- **`t in _PROFANITY_LEXICON`:** frozenset membership check -> $O(1)$ per token (hash lookup)
- Sum over $W$ tokens -> $O(L)$

**Per row: $O(L)$; dominating operation: regex tokenisation**


## 4. Obfuscated Profanity Feature (1 feature)
`obfuscated_profanity_count`

```python
# build_features.py, lines 295-304
_LEETSPEAK_MAP = str.maketrans({"@": "a", "$": "s", ...})  # built once at module level

def _normalize_leetspeak(token: str) -> str:
    return re.sub(r"[^a-z]", "", token.lower().translate(_LEETSPEAK_MAP))  # O(|token|)

def _obfuscated_profanity_count(text: str) -> int:
    count = 0
    for raw in text.split():                              # O(L) iterations
        plain      = re.sub(r"[^a-z]", "", raw.lower())  # O(|token|)
        normalised = _normalize_leetspeak(raw)            # O(|token|)
        if normalised in _PROFANITY_LEXICON and plain not in _PROFANITY_LEXICON:  # O(1)
            count += 1
    return count
```

- **`text.split()`:** $O(L)$ to produce tokens
- **Per token:** `str.translate` (1-pass substitution table) -> $O(|\text{token}|)$; `re.sub` to strip non-alpha -> $O(|\text{token}|)$; two (frozenset) lookups -> $O(1)$ 
- Summed over all tokens -> $O(L)$

**Per row: $O(L)$; dominant operation: token loop with translate + regex**


## 5. Slang Feature (1 feature)
`slang_count`

```python
# build_features.py, lines 308-310
def _slang_count(text: str) -> int:
    words = re.findall(r"\b\w+\b", text.lower())           # tokenise: O(L)
    return sum(1 for w in words if w in _TOXIC_SLANG)       # frozenset lookup: O(1) each
```

- **`re.findall`:** one regex scan over $L$ characters -> $O(L)$
- **`w in _TOXIC_SLANG`:** frozenset hash lookup -> $O(1)$ per token
- Sum over tokens -> $O(L)$

**Per row: $O(L)$; dominant operation => regex scan**


## 6. Text Shape Features (4 features)
`char_count`, `word_count`, `exclamation_count`, `uppercase_ratio`

```python
# build_features.py, lines 414-419
words = text.split()                          # O(L)
row["char_count"]        = len(text)          # O(1) - Python stores length internally
row["word_count"]        = len(words)         # O(1) - len of already-computed list
row["exclamation_count"] = text.count("!")   # O(L) - linear scan for '!'
row["uppercase_ratio"]   = _uppercase_ratio(text)

# build_features.py, lines 314-319
def _uppercase_ratio(text: str) -> float:
    words = text.split()                              # O(L) - called again here
    upper = [w for w in words if w.isupper() and len(w) > 1]  # O(L)
    return len(upper) / len(words)                    # O(1)
```

- **`len(text)`:** $O(1)$ - Python's `str` stores the length as an attribute
- **`text.count("!")`:** linear scan over $L$ characters -> $O(L)$
- **`text.split()`:** $O(L)$ - called twice (here and inside `_uppercase_ratio`); not shared
- **`w.isupper()`:** $O(|w|)$ per word, summed -> $O(L)$

**Per row: $O(L)$; dominant operation: `text.split()` + `isupper()` scan**


## 7. Unique Word Ratio (1 feature)
`unique_word_ratio`

```python
# build_features.py, lines 323-327
def _unique_word_ratio(text: str) -> float:
    words = text.split()             # O(L)
    return len(set(words)) / len(words)  # set construction: O(L), len: O(1)
```

- **`text.split()`:** $O(L)$ to produce tokens
- **`set(words)`:** inserts all tokens into a hash set -> $O(L)$ 

**Per row: $O(L)$; dominant operation => set construction**

---

## 8. Elongation Features (2 features)
`elongated_token_count`, `consecutive_punct_count`

```python
# build_features.py, lines 331-335
_ELONGATE_RE = re.compile(r"(.)\1{2,}", re.IGNORECASE)  # compiled once - line 118
_PUNCT_RE    = re.compile(r"[^\w\s]{2,}")               # compiled once - line 122

def _elongated_token_count(text: str) -> int:
    return sum(1 for tok in text.split() if _ELONGATE_RE.search(tok))  # O(L)

def _consecutive_punct_count(text: str) -> int:
    return len(_PUNCT_RE.findall(text))   # one regex scan: O(L)
```

- **`_elongated_token_count`:** `text.split()` -> $O(L)$; `_ELONGATE_RE.search(tok)` per token -> $O(|\text{tok}|)$ each -> total $O(L)$
- **`_consecutive_punct_count`:** `_PUNCT_RE.findall(text)` -> single scan over $L$ characters -> $O(L)$

**Per row: $O(L)$; dominant operation => regex scan**



## 9. URL / IP Features (3 features)
`url_count`, `ip_count`, `has_url_or_ip`

```python
# build_features.py, lines 339-343
_URL_RE  = re.compile(r"(?:https?://[^\s]+)|(www\.[^\s]+)", re.IGNORECASE)  # compiled once - line 99
_IPV4_RE = re.compile(r"\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}"
                      r"(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b")       # compiled once - line 103

def _url_count(text: str) -> int:
    return len(_URL_RE.findall(text))    # one scan: O(L)

def _ip_count(text: str) -> int:
    return len(_IPV4_RE.findall(text))   # one scan: O(L)

# build_features.py, lines 428-432
url_n = _url_count(text)
ip_n  = _ip_count(text)
row["has_url_or_ip"] = int(url_n > 0 or ip_n > 0)   # O(1)
```

- **`_URL_RE.findall(text)`:** one linear scan over $L$ characters -> $O(L)$
- **`_IPV4_RE.findall(text)`:** one linear scan over $L$ characters -> $O(L)$; more alternation branches but still one pass
- **`has_url_or_ip`:** $O(1)$ boolean from already-computed counts

**Per row: $O(L)$; dominant operation => two regex scans**



## 10. Syntactic Features (3 features)
`negation_count`, `sentence_count`, `avg_sentence_length`

```python
# build_features.py, lines 347-359
_NEGATION_RE = re.compile(r"\b\w+(?:'\w+)?\b")  # compiled once - line 114
_SENTENCE_RE = re.compile(r"[.!?]+")            # compiled once - line 126

def _negation_count(text: str) -> int:
    tokens = _NEGATION_RE.findall(text.lower())           # O(L)
    return sum(1 for t in tokens if t in _NEGATION_WORDS) # frozenset: O(1) each

def _sentence_count(text: str) -> int:
    sents = [s for s in _SENTENCE_RE.split(text.strip()) if s.strip()]  # O(L)
    return len(sents)   # O(1)

def _avg_sentence_length(text: str) -> float:
    sents = [s for s in _SENTENCE_RE.split(text.strip()) if s.strip()]  # O(L)
    return sum(len(s.split()) for s in sents) / len(sents)  # O(L) total
```

- **`_negation_count`:** `_NEGATION_RE.findall` -> $O(L)$; frozenset lookup per token -> $O(1)$ each
- **`_sentence_count`:** `_SENTENCE_RE.split` -> $O(L)$
- **`_avg_sentence_length`:** same split -> $O(L)$; `s.split()` per sentence -> $O(L)$ total across all sentences
- Note: `_SENTENCE_RE.split` no result sharing -> called seperatly

**Per row: $O(L)$; dominant operation =>  regex scan + token loop**



## 11. Identity Features (7 features)
`identity_mention_count`, `identity_race`, `identity_gender`, `identity_sexuality`,
`identity_religion`, `identity_disability`, `identity_nationality`

```python
# build_features.py, lines 363-368
_IDENTITY_ALL_RE: re.Pattern  # combined pattern over all categories - compiled once, line 247
_IDENTITY_PATTERNS: dict      # one pattern per category - compiled once, lines 241-244

def _extract_identity(text: str) -> dict:
    total   = len(_IDENTITY_ALL_RE.findall(text))     # one combined scan: O(L)
    results = {"identity_mention_count": total}
    for cat, pattern in _IDENTITY_PATTERNS.items():   # 6 categories, each O(L)
        results[f"identity_{cat}"] = int(bool(pattern.search(text)))
    return results
```

- **`_IDENTITY_ALL_RE.findall(text)`:** one scan over $L$ characters using a single compiled alternation pattern -> $O(L)$
- **`pattern.search(text)`:** 6 category-specific regex scans, each $O(L)$ - 6 is a constant, drops out -> $O(L)$
- All 7 patterns compiled once at module level (lines 241-244) 

**Per row: $O(L)$; dominant operation: 7 regex scans (most scans of any single feature group)**



## 12. StandardScaler - `_common.py`, lines 78-79

```python
# _common.py, lines 78-79
X_train_feat = scaler.fit_transform(X_train_dense)   # O(n·p)
X_test_feat  = scaler.transform(X_test_dense)         # O(n·p)
```

- **`fit`:** one pass over the $n \times p$ matrix to compute column-wise mean and std -> $O(n \cdot p)$
- **`transform`:** element-wise subtract mean, divide by std for every cell -> $O(n \cdot p)$

**Time: $O(n \cdot p)$ - Space: $O(p)$** (stores only $p$ means and $p$ stds)

---

## Combined Analysis

### Per-feature summary (single row)

| # | Feature group | Features | Per-row cost | Dominant operation |
|---|---------------|----------|-------------|--------------------|
| 1 | Sentiment | 6 | $O(L)$ | VADER token-by-token lexicon lookup |
| 2 | Second-person | 3 | $O(L)$ | `_SECOND_PERSON_RE.findall` |
| 3 | Profanity | 1 | $O(L)$ | `_WORD_RE.findall` + frozenset loop |
| 4 | Obfuscated profanity | 1 | $O(L)$ | token loop + `str.translate` |
| 5 | Slang | 1 | $O(L)$ | `re.findall` + frozenset loop |
| 6 | Text shape | 4 | $O(L)$ | `text.split()` + `isupper()` scan |
| 7 | Unique word ratio | 1 | $O(L)$ | `set(words)` construction |
| 8 | Elongation | 2 | $O(L)$ | two regex scans (`_ELONGATE_RE`, `_PUNCT_RE`) |
| 9 | URL / IP | 3 | $O(L)$ | two regex scans (`_URL_RE`, `_IPV4_RE`) |
| 10 | Syntactic | 3 | $O(L)$ | `_SENTENCE_RE.split` + token loop |
| 11 | Identity | 7 | $O(L)$ | 7 compiled regex scans |
| | **Total per row** | **32** | $O(L)$ | all groups $O(L)$; $p = 32$ is a constant and drops out |

### Total over all $n$ rows

```python
# build_features.py, lines 401-441
for text in texts:           # O(n) outer loop
    # 11 feature groups - each O(L) per row, p=32 constant drops out
    ...
```

- All 11 feature groups are $O(L)$ per row
- $p = 32$ is a constant -> $O(p \cdot L) = O(L)$ per row
- Outer loop over $n$ rows -> **total: $O(n \cdot L)$**
- StandardScaler adds $O(n \cdot p) = O(n)$ - dominated by the feature transform

**Total pipeline: $O(n \cdot L)$**

**Space: $O(n \cdot p) = O(n)$** - the $n \times 32$ output matrix; all row-level intermediates are discarded per iteration

### What dominates?

- Within a single row all groups are $O(L)$; the constant hidden inside differs (VADER does more work per character than a plain `len()`) but the asymptotic class is the same
- Across all rows the outer loop over $n$ is the dominant factor
- `precompute_features()` in `_common.py` (lines 54-91) caches the result - the $O(n \cdot L)$ cost is paid **exactly once**; every model run after that skips it entirely